In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG workspace")

# Data limite superior dos recortes de "últimos 2 e 5 anos":
# lançamento mais recente da base, ignorando filmes não lançados e datas futuras.
data_max = spark.sql("""
    SELECT MAX(data_lancamento)
    FROM gold.dim_movies
    WHERE status_filme = 'Lançado'
      AND data_lancamento IS NOT NULL
      AND data_lancamento <= current_date()
""").first()[0]
print("Data limite (lançamento mais recente):", data_max)

Data limite (lançamento mais recente): 2026-02-19


In [0]:
# 1. Receita total (R$) somada de todos os filmes da base.
# SUM ignora NULL: filmes sem receita informada não entram na soma.
display(spark.sql("""
    SELECT SUM(receita_brl)                           AS receita_total_brl,
           SUM(receita_usd)                           AS receita_total_usd,
           COUNT(receita_brl)                         AS filmes_com_receita
    FROM gold.fact_movies_performance
"""))

receita_total_brl,receita_total_usd,filmes_com_receita
802407675044.81,161476228577.00,3248


In [0]:
# 2. Os 5 filmes com maior popularidade (título e valor).
display(spark.sql("""
    SELECT m.titulo, f.popularidade
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m ON f.sk_movie_id = m.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
Battipaglia 1969,1969.0


In [0]:
# 3. Quantidade de filmes por gênero, do maior para o menor volume.
# Passa pela tabela-ponte; COUNT(DISTINCT) garante que cada filme conta uma vez por gênero.
display(spark.sql("""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM gold.bridge_movie_genre b
    JOIN gold.dim_genres g ON b.sk_genre_id = g.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC
"""))

nome_genero,qtd_filmes
Drama,32127
Documentary,18927
Comedy,18536
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
Tv Movie,4066


In [0]:
# 4. Os 10 filmes de maior receita, com receita em US$ e R$ e posição via RANK().
# RANK() deixa empates na mesma posição (por isso pode haver mais de 10 linhas em caso de empate).
display(spark.sql("""
    SELECT titulo, receita_usd, receita_brl, posicao
    FROM (
        SELECT m.titulo, f.receita_usd, f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao
        FROM gold.fact_movies_performance f
        JOIN gold.dim_movies m ON f.sk_movie_id = m.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    ) t
    WHERE posicao <= 10
    ORDER BY posicao
"""))

titulo,receita_usd,receita_brl,posicao
Avengers: Endgame,2800000000.00,13913760000.00,1
Avatar: The Way of Water,2320250281.00,11529787696.35,2
AVENGERS: INFINITY WAR,2052415039.00,10198860811.80,3
spider-man: no way home,1921847111.00,9550042663.98,4
The Lion King,1663075401.00,8264154282.65,5
Top Gun: Maverick,1488732821.00,7397811134.11,6
Barbie,1428545028.00,7098725953.14,7
The Super Mario Bros. Movie,1355725263.00,6736869976.90,8
Black Panther,1349926083.00,6708052691.64,9
Star Wars: The Last Jedi,1332698830.00,6622447026.04,10


In [0]:
# 5. Ator com maior quantidade de participações em filmes lançados nos últimos 2 anos,
# contados a partir da data limite (não da data de hoje).
# Janela: (data_max - 24 meses, data_max]. Só tipo_pessoa = 'Ator'.
display(spark.sql(f"""
    SELECT p.nome_pessoa AS ator, COUNT(DISTINCT m.sk_movie_id) AS qtd_filmes
    FROM gold.dim_movies m
    JOIN gold.bridge_movie_person b ON m.sk_movie_id = b.sk_movie_id
    JOIN gold.dim_people p          ON b.sk_person_id = p.sk_person_id
    WHERE p.tipo_pessoa = 'Ator'
      AND m.status_filme = 'Lançado'
      AND m.data_lancamento >  add_months(DATE'{data_max}', -24)
      AND m.data_lancamento <= DATE'{data_max}'
    GROUP BY p.nome_pessoa
    ORDER BY qtd_filmes DESC
    LIMIT 5
"""))

ator,qtd_filmes
Kevin Hart,63
Nathalie Emmanuel,56
Ben Schwartz,53
John Cena,50
Paula Pell,45


In [0]:
# 6. Produtora com maior lucro nos últimos 5 anos, contados a partir da data limite.
# Janela: (data_max - 60 meses, data_max]. Lucro = soma de lucro_usd dos filmes da produtora.
# Filmes sem orçamento ou receita têm lucro NULL e não entram na soma (decisão documentada).
# Um filme com várias produtoras conta o seu lucro inteiro para cada uma (relação N:N via bridge).
display(spark.sql(f"""
    SELECT c.nome_produtora,
           SUM(f.lucro_usd)           AS lucro_total_usd,
           SUM(f.lucro_brl)           AS lucro_total_brl,
           COUNT(f.lucro_usd)         AS filmes_com_lucro
    FROM gold.dim_movies m
    JOIN gold.fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
    JOIN gold.bridge_movie_company b    ON m.sk_movie_id = b.sk_movie_id
    JOIN gold.dim_companies c           ON b.sk_company_id = c.sk_company_id
    WHERE m.data_lancamento >  add_months(DATE'{data_max}', -60)
      AND m.data_lancamento <= DATE'{data_max}'
    GROUP BY c.nome_produtora
    HAVING COUNT(f.lucro_usd) > 0
    ORDER BY lucro_total_usd DESC
    LIMIT 5
"""))

nome_produtora,lucro_total_usd,lucro_total_brl,filmes_com_lucro
Universal Pictures,5772329679.00,28683860640.89,24
Marvel Studios,4953462823.00,24614747460.07,8
Columbia Pictures,3662050755.00,18197462611.75,12
Pascal Pictures,2701952454.00,13426542134.42,3
Illumination,2431353473.00,12081881678.03,3
